# RecyMind – Semaine 2 : fine-tuning et comparaison des modèles
Personne 2 · MobileNetV3Small contre EfficientNetB0, sur les 7 classes du contrat.

**Dossiers attendus sur Drive** (fournis par Personne 1) :
```
recymind/data_v2/train/<classe>/     dataset regroupé + photos de l'équipe (entraînement)
recymind/data_v2/val/<classe>/       validation
recymind/data_v2/test_equipe/<classe>/   photos de l'équipe JAMAIS utilisées pour entraîner
```
Classes : plastique, papier_carton, verre, metal, textile, autre, tapis_vide. GPU activé obligatoire.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf, json, time, numpy as np
BASE = "/content/drive/MyDrive/recymind/data_v2"
IMG, BATCH = (224, 224), 32

def charger(sous_dossier, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        f"{BASE}/{sous_dossier}", image_size=IMG, batch_size=BATCH, shuffle=shuffle, seed=42)

train_ds, val_ds, test_ds = charger("train", True), charger("val", False), charger("test_equipe", False)
class_names = train_ds.class_names
print(class_names)
assert class_names == sorted(["plastique","papier_carton","verre","metal","textile","autre","tapis_vide"]), "Classes inattendues"
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

## Construction : augmentation + base pré-entraînée + tête
L'augmentation (retournement, rotation, luminosité, contraste) n'agit qu'à l'entraînement.

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.2),
])

def construire(nom):
    if nom == "mobilenetv3":
        base = tf.keras.applications.MobileNetV3Small(input_shape=IMG+(3,), include_top=False, weights="imagenet")
    else:
        base = tf.keras.applications.EfficientNetB0(input_shape=IMG+(3,), include_top=False, weights="imagenet")
    base.trainable = False
    entree = tf.keras.Input(IMG + (3,))      # pixels entre 0 et 255 (le prétraitement est dans la base)
    x = augment(entree)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    sortie = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
    return tf.keras.Model(entree, sortie), base

def entrainer(nom):
    model, base = construire(nom)
    # Phase 1 : on entraîne seulement la tête
    model.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", ["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=5)
    # Phase 2 : fine-tuning des 30 dernières couches avec un learning rate faible
    base.trainable = True
    for couche in base.layers[:-30]:
        couche.trainable = False
    model.compile(tf.keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", ["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=5)
    return model

## Entraînement des deux modèles

In [ ]:
modeles = {nom: entrainer(nom) for nom in ["mobilenetv3", "efficientnetb0"]}

## Comparaison : accuracy et temps d'inférence
Le choix se fait sur le **lot test de l'équipe** (jamais vu), pas sur la validation.

In [ ]:
def temps_ms(model, n=50):
    img = np.random.randint(0, 255, (1,)+IMG+(3,)).astype("float32")
    model.predict(img, verbose=0)                       # échauffement
    t = time.time()
    for _ in range(n): model.predict(img, verbose=0)
    return (time.time() - t) / n * 1000

resultats = {}
for nom, m in modeles.items():
    _, acc_val = m.evaluate(val_ds, verbose=0)
    _, acc_test = m.evaluate(test_ds, verbose=0)
    resultats[nom] = {"acc_val": round(acc_val, 3), "acc_test_equipe": round(acc_test, 3), "ms_par_image": round(temps_ms(m), 1)}
    print(nom, resultats[nom])

## Meilleur modèle : sauvegarde et matrice de confusion

In [ ]:
meilleur = max(resultats, key=lambda n: resultats[n]["acc_test_equipe"])
print("Meilleur modèle :", meilleur)
dossier = "/content/drive/MyDrive/recymind/modeles"
!mkdir -p "{dossier}"
modeles[meilleur].save(f"{dossier}/modele_v2_{meilleur}.keras")
json.dump(class_names, open(f"{dossier}/classes_v2.json", "w"))
json.dump(resultats, open(f"{dossier}/comparaison_v2.json", "w"), indent=1)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
y_vrai = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(modeles[meilleur].predict(test_ds, verbose=0), axis=1)
ConfusionMatrixDisplay(confusion_matrix(y_vrai, y_pred), display_labels=class_names).plot(xticks_rotation=45)
plt.title(f"Matrice de confusion – {meilleur} (photos de l'équipe)"); plt.show()